In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

**References**

- [Large Language Models can Self-Improve](https://arxiv.org/pdf/2210.11610.pdf)

# The Self-Improvement loop

The simplest version of Synthetic Data generation is to
- start with a small quantity of "seed" examples
- use these to generate additional examples
    - e.g., In context learning

But why stop there ? Why not 
- *augment* the seed examples
- with the synthetic examples

to create an enlarged set of seed examples
- which are used to create even more examples



This iterative process is called *Self-Improvement*

We write this as


$$\model_{(\tt+1)} = \text{Train}(\model_\tp, \text{Filter}(\model_\tp(\text{Noise}))) $$

Starting with base model $\model_{(0)}$
- fine-tune $\model_{(0)}$
- using a mixture of strong (human-generated) and weak (LLM generated) fine-tuning  examples of the Target task

to create a weak model $\model_{(1)}$.

$\model_{(1)}$ can be used to generated more examples
- which augment the training examples

which are used to fine tune $\model_{(1)}$ into stronger model $\model_{(2)}$.

The synthetic examples created at each step are *filtered*
- so that only the highest quality examples
- are used in the augmentation


# Case study: Self-Improvement of Backtranslation

Backtranslation converts a model $M_{xy}$
- predict $\y = \text{Response}$ 
- as the continuation of prompt $\x = \langle \text{Instruction}, \text{Context} \rangle$
 

into model $M_{yx}$
- predict $\langle \text{Instruction}, \text{Context} \rangle$
- as the continuation of prompt $\y$

The Self-Improvement process
- treats $\model_{(0)} = \model_{yx}$

This weak model generates synthetic examples
- which are *candidates* for augmenting the seed dataset

Candidates are selected for augmentation
- via a process that filters for quality

Here is the workflow:

<table>
    <center><strong>Instruction Backtranslation</strong></center>
    <tr>
        <img src="images/instruction_backtranslation.png" width=70%>
    </tr>
    
Attribution: https://arxiv.org/pdf/2308.06259.pdf#page=2
</table>

## Filtering: Majority voting 

Majority voting is technique that
- takes advantage of non-deterministic sampling to create an LLM output
- in order to create multiple responses to the same prompt

and selects the "highest quality" response as the one that appears most frequently.
 majority voting to select the "best" response

Here is an illustration of using Majority Voting to find the best answer to a question.
<br>

<table>
    <center><strong>Self Improvement of an LLM</strong></center>
    <tr>
        <img src="images/llm_self_improvement.png">
    </tr>
    
Attribution: https://arxiv.org/pdf/2210.11610.pdf#page=2
</table>

How can we use Majority Voting to rate the quality of the $\langle \text{Instruction}, \text{Context} \rangle$ pairs ?

Recall how the "predict the next token" LLM works
- output token $\y_\tp$ of the response is generated conditionally, based on the preceding tokens $\y_{([0 .. \tt-1])}$
- by creating a probability distribution 
$$\prc{ \y_\tp}{\y_{([0 .. \tt-1])}}$$
- and sampling one token $\y_\tp$ 

Let $\pr{\tt}$ denote the probability of the actual token chosen at position $\tt$

Then the likelihood of full response $\y = \y_{([0..T])}$ is
$$\pr{\y} = \prod_{\tt=0}^T \pr{\tt}$$

The sampling does *not have to be deterministic*
- non-zero temperature

so we can generate multiple responses to a prompt
- and use the Likelihood of the response

as a measure of quality.

In the Synthetic Instruction Following example case
- use the same $$\y = \text{Response}$$ 
- to generate multiple $$\x - \langle \text{Instruction}, \text{Context} \rangle $$ candidates
- and filter candidates to those with high likelihood

## Filtering: LLM as a Judge

There is a very simple way to rate the quality of a synthetic example ?

Prompt an LLM to do it !

This technique is using an *LLM as a Judge*
- an increasing common technique
- requiring a strong and broad LLM as Judge

The following prompt requests that the LLM evaluate the
synthetic example using a rating scale of $1$ (low quality) to $5$ (high quality)

<br>
<table>
    <center><strong>Instruction Backtranslation Curation</strong></center>
    <tr>
        <img src="images/instruction_backtranslation_curating.png" width=70%>
    </tr>
    
Attribution: https://arxiv.org/pdf/2308.06259.pdf#page=4
</table>

A very large general-purpose LLM/Assistant
- can serve as a good basis for a Judge
- for a specialized task

due to 
- its ability to follow precise judging instructions (acquired during post-training)
- its broad knowledge, acquired during pre-training
- and render informed judgements

# Discussion

Key concepts
- Self-improvement
    - A kind of bootstrapping
    
- LLM as a Judge

In [2]:
print("Done")

Done
